# Training on Google Colab

Colab is a **different computer** — it cannot see your local drive. So this notebook:

1. pulls the **code** in from GitHub
2. pulls the **data** in by upload (the small processed arrays, not raw RadioML/RadChar)
3. trains (ensemble + single model)
4. pushes **everything** back out to your machine

## Read this before you start

**Colab's disk is temporary.** Everything under `/content/` is deleted when the
session ends — idle timeout, or ~12 hours maximum. If you train for 20 minutes
and close the tab without running the download cell, the model is gone.

**Enable the GPU first:** Runtime → Change runtime type → Hardware accelerator → GPU.
Do this *before* running anything; switching later restarts the session and
wipes your uploads.

**Run cells top to bottom, in order.** This version fixes an ordering bug from
before: sanity-check now runs *before* any training, and the dataset is never
rebuilt inside Colab (that was silently corrupting the uploaded data).

## 1. Get the code

If the repo is private, this fails. Either make it public, or upload a zip of the repo instead.

**`eavan-train-overall` is the integration branch** — it now has the RadioML
civilian loader merged in (from `eileen-civilian`), so it matches the 7-class
dataset you upload below. Update `-b <branch>` if that changes.

In [ ]:
%cd /content
!rm -rf sedicAI_NEXA
!git clone -b eavan-train-overall https://github.com/eavan127/sedicAI_NEXA.git
%cd /content/sedicAI_NEXA
!pwd

In [ ]:
# Colab already has torch, numpy, scipy, sklearn, matplotlib.
# Only these are missing:
!pip install -q pyyaml h5py

## 2. Check the GPU is actually attached

If this says `CUDA: False`, you skipped the Runtime → Change runtime type step.
Training still works on CPU, just much slower.

In [ ]:
import torch
print('CUDA:', torch.cuda.is_available())
print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU only')

## 3. Get the data in

Upload the three arrays from your machine:

    data/processed/X.npy
    data/processed/y.npy
    data/processed/snr_labels.npy

About 29 MB total — seconds to upload.

**Upload the processed arrays, not RadChar/RadioML.** Those raw sources are gigabytes,
and this notebook never rebuilds the dataset in Colab — rebuilding without the raw
files present would silently overwrite these uploads with an incomplete dataset
(missing civilian classes). If you need to rebuild, do it locally with
`python -m src.data.build_dataset`, then re-upload the three arrays here.

**Use this cell OR the Drive cell below, not both.**

In [ ]:
import os, shutil
from google.colab import files
os.makedirs('data/processed', exist_ok=True)
print('Select X.npy, y.npy and snr_labels.npy (you can pick all three at once)')
uploaded = files.upload()
for name in uploaded:
    shutil.move(name, f'data/processed/{name}')
!ls -la data/processed/

### Alternative: mount Google Drive

Better if you will run this repeatedly — the files persist between sessions, so
you upload once instead of every time. Put the arrays in a `sedic/` folder in
your Drive first.

**Commented out on purpose** — this is an alternative to the upload cell above,
not an extra step. Uncomment and run *instead of* the upload cell, not after it.

In [ ]:
# from google.colab import drive
# drive.mount('/content/drive')
# !mkdir -p data/processed
# !cp /content/drive/MyDrive/sedic/*.npy data/processed/

## 4. Sanity check — BEFORE spending any GPU time

Confirms the code runs and the data loaded correctly. Takes seconds.
**If this fails, stop — do not run the training cells below.**

In [ ]:
!python -m pytest -q

import numpy as np
X = np.load('data/processed/X.npy')
y = np.load('data/processed/y.npy')
print('X:', X.shape, X.dtype)
print('class counts:', np.bincount(y))

## 5. Train

Two runs, in this order:

1. **Ensemble** (`train_ensemble.py`) — 5 seeds, averaged. This is your main
   robustness result, since single-seed recall was measured to swing ~2–9 points
   on the judged classes.
2. **Variance measurement** — quantifies that swing directly. Prints only, no
   file is written, so **copy the printed spread numbers somewhere before you
   move on** (screenshot or paste into your notes).
3. **Single model** (`src.train`) — a plain baseline run, useful to quote
   alongside the ensemble result in the brief.

Watch `val_loss` in the single-model run: falling means learning; rising while
`train_loss` falls means overfitting (harmless here — only the best checkpoint
is kept).

In [ ]:
!python scripts/train_ensemble.py --models 5

In [ ]:
!python scripts/measure_variance.py --runs 5

In [ ]:
!python -m src.train

In [ ]:
!python -m src.evaluate

### Jamming sub-type breakdown (barrage / tone / sweep)

Not part of the scored benchmark — this is a diagnostic probe that shows which
kind of jamming (wideband noise, single-tone, or sweeping chirp) the model
handles best, and saves a chart + JSON for the technical brief.

In [ ]:
!python -m src.data.diagnose_jamming

### Preview the plots before downloading

In [ ]:
from IPython.display import Image, display
display(Image('evals/confusion_matrix.png'))
display(Image('evals/accuracy_vs_snr.png'))
display(Image('evals/jamming_subtypes.png'))

## 6. Get everything out — DO NOT SKIP THIS

This is the step people forget. Everything above is deleted when the session
ends. This now includes the **ensemble checkpoints, ensemble scorecard, and
jamming sub-type breakdown** — the single-model download list from before was
silently losing these.

Save the downloads into `results/` and `evals/` in your local repo.

In [ ]:
from google.colab import files
import os, glob

paths = [
    'results/best_model.pt',
    'evals/scorecard.json',
    'evals/confusion_matrix.png',
    'evals/accuracy_vs_snr.png',
    'evals/ensemble_scorecard.json',
    'evals/jamming_subtypes.json',
    'evals/jamming_subtypes.png',
] + sorted(glob.glob('results/ensemble_*.pt'))

for path in paths:
    if os.path.exists(path):
        files.download(path)
    else:
        print('missing:', path)

### Or save straight to Drive (survives the session)

In [ ]:
# !mkdir -p /content/drive/MyDrive/sedic/runs
# !cp results/best_model.pt results/ensemble_*.pt evals/*.json evals/*.png /content/drive/MyDrive/sedic/runs/